### Gold layer: match_predictions

Home/Draw/Away probabilities per match from Dixon-Coles.
Each match uses the ratings of its own matchday (point-in-time, no lookahead).
In-memory only until the metrics look right.

In [6]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\arnol\OneDrive\Documentos\Progaming\Datascience\datascience\football_predictor")
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.parquet_io import read_league_all_seasons
from utils.dixon_coles import outcome_probs

LAKE_ROOT = PROJECT_ROOT / "data"
LEAGUES = ["laliga", "primeira_liga"]

silver = {lg: read_league_all_seasons(LAKE_ROOT, layer="silver", league=lg).to_pandas()
          for lg in LEAGUES}
ratings = {lg: read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                       filename="team_ratings.parquet").to_pandas()
           for lg in LEAGUES}

for lg in LEAGUES:
    print(lg, "silver:", silver[lg].shape, "| ratings:", ratings[lg].shape)

laliga silver: (1520, 19) | ratings: (2420, 17)
primeira_liga silver: (1224, 19) | ratings: (1962, 17)


In [7]:
def build_predictions(matches, rat):
    rt = rat[["season", "matchday", "team_key", "attack", "defense"]]
    par = (rat.groupby(["season", "matchday"])
              [["mu", "gamma", "rho", "as_of", "is_warmup", "is_provisional"]]
              .first().reset_index())

    m = matches.merge(
        rt.rename(columns={"team_key": "home_team_key", "attack": "h_att", "defense": "h_def"}),
        on=["season", "matchday", "home_team_key"], how="inner")
    m = m.merge(
        rt.rename(columns={"team_key": "away_team_key", "attack": "a_att", "defense": "a_def"}),
        on=["season", "matchday", "away_team_key"], how="inner")
    m = m.merge(par, on=["season", "matchday"], how="inner")

    m["lambda_home"] = np.exp(m["mu"] + m["gamma"] + m["h_att"] - m["a_def"])
    m["lambda_away"] = np.exp(m["mu"] + m["a_att"] - m["h_def"])
    probs = [outcome_probs(a, b, c) for a, b, c in zip(m["lambda_home"], m["lambda_away"], m["rho"])]
    m[["p_home", "p_draw", "p_away"]] = np.array(probs)
    return m


preds = {lg: build_predictions(silver[lg], ratings[lg]) for lg in LEAGUES}

for lg, p in preds.items():
    print(f"=== {lg} === rows: {len(p)}")
    print("prob sum min/max:", p[["p_home", "p_draw", "p_away"]].sum(axis=1).agg(["min", "max"]).round(4).tolist())
    print("mean probs H/D/A:", p[["p_home", "p_draw", "p_away"]].mean().round(3).tolist())

    ev = p[(p["status"] == "FINISHED") & ~p["is_warmup"]]
    y = ev["result"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
    P = ev[["p_home", "p_draw", "p_away"]].to_numpy()
    freq = np.bincount(y, minlength=3) / len(y)
    print("evaluated matches:", len(ev), "(warmup excluded)")
    print("actual freq H/D/A:", freq.round(3).tolist())
    print("accuracy:", round(float((P.argmax(axis=1) == y).mean()), 3))
    print("log-loss model:", round(float(-np.log(P[np.arange(len(y)), y]).mean()), 4),
          "| baseline (global freq):", round(float(-np.log(freq[y]).mean()), 4))
    print()

=== laliga === rows: 1210
prob sum min/max: [1.0, 1.0]
mean probs H/D/A: [0.438, 0.261, 0.301]
evaluated matches: 1080 (warmup excluded)
actual freq H/D/A: [0.461, 0.258, 0.281]
accuracy: 0.526
log-loss model: 0.9794 | baseline (global freq): 1.0632

=== primeira_liga === rows: 981
prob sum min/max: [1.0, 1.0]
mean probs H/D/A: [0.441, 0.236, 0.323]
evaluated matches: 860 (warmup excluded)
actual freq H/D/A: [0.422, 0.266, 0.312]
accuracy: 0.547
log-loss model: 0.9409 | baseline (global freq): 1.0798



### Write match_predictions to Gold

Same freeze rule as team_ratings: a match's prediction is fixed once its
matchday is no longer provisional, so a finished match never gets
"predicted" with hindsight ratings.

In [8]:
from utils.parquet_io import read_partition, write_partition
from utils.schema import MATCH_PREDICTIONS_SCHEMA
import pyarrow as pa

COLS = [f.name for f in MATCH_PREDICTIONS_SCHEMA]
KEY = ["league", "season", "matchday", "home_team_key", "away_team_key"]

final = {}
for lg, p in preds.items():
    p = p.copy()
    p["is_provisional"] = p["is_provisional"] if "is_provisional" in p else p["status"] != "FINISHED"
    final[lg] = p[COLS]
    print(f"=== {lg} ===")
    print("rows:", len(p), "| duplicate keys:", int(p.duplicated(KEY).sum()))
    print("nulls (excl. result):", int(p[[c for c in COLS if c != "result"]].isna().sum().sum()))
    print("null result count:", int(p["result"].isna().sum()), "| non-FINISHED count:", int((p["status"] != "FINISHED").sum()))
    print()

=== laliga ===
rows: 1210 | duplicate keys: 0
nulls (excl. result): 0
null result count: 20 | non-FINISHED count: 20

=== primeira_liga ===
rows: 981 | duplicate keys: 0
nulls (excl. result): 0
null result count: 13 | non-FINISHED count: 13



In [12]:
def write_predictions(df, lake_root):
    for (lg, season), part in df.groupby(["league", "season"]):
        path = lake_root / "gold" / lg / f"season={season}" / "match_predictions.parquet"
        if path.exists():
            old = read_partition(path).to_pandas().drop(columns=["_ingested_at"], errors="ignore")
            frozen = old[~old["is_provisional"]]
            key_cols = ["matchday", "home_team_key", "away_team_key"]
            new = part[~part.set_index(key_cols).index.isin(frozen.set_index(key_cols).index)]
            part = pd.concat([frozen, new], ignore_index=True)
        part = part.sort_values(["matchday", "match_date"])
        table = pa.Table.from_pandas(part[COLS], schema=MATCH_PREDICTIONS_SCHEMA, preserve_index=False)
        write_partition(table, path)
        print(f"{lg} {season}: {len(part)} rows")

for lg in LEAGUES:
    write_predictions(final[lg], LAKE_ROOT)

laliga 2023-2024: 380 rows
laliga 2024-2025: 380 rows
laliga 2025-2026: 380 rows
laliga 2026-2027: 70 rows
primeira_liga 2023-2024: 306 rows
primeira_liga 2024-2025: 306 rows
primeira_liga 2025-2026: 306 rows
primeira_liga 2026-2027: 63 rows


In [10]:
for lg in LEAGUES:
    g = read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                filename="match_predictions.parquet").to_pandas()
    print(lg, g.shape, "| duplicate keys:", int(g.duplicated(["season", "matchday", "home_team_key", "away_team_key"]).sum()))

laliga (1210, 17) | duplicate keys: 0
primeira_liga (981, 17) | duplicate keys: 0
